In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col,
    rank,
    dense_rank,
    row_number
)

spark = SparkSession.builder \
    .appName("Ranking and Aggregations Demo") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Version:", spark.version)
print ("Spark Session Created Successfully!")

In [ ]:
employee_data = [
    (101, "Amit",   "IT",      75000),
    (102, "Neha",   "IT",      65000),
    (103, "Rahul",  "IT",      65000),
    (104, "Priya",  "IT",      55000),

    (105, "Karan",  "HR",      70000),
    (106, "Sneha",  "HR",      60000),
    (107, "Rohit",  "HR",      60000),
    (108, "Pooja",  "HR",      50000),

    (109, "Arjun",  "Finance", 90000),
    (110, "Meera",  "Finance", 80000),
    (111, "Vijay",  "Finance", 80000),
    (112, "Anjali", "Finance", 70000)
]

columns = [
    "emp_id",
    "emp_name",
    "department",
    "salary"
]

employees_df = spark.createDataFrame(
    employee_data,
    columns
)

employees_df.show()

In [ ]:
salary_window = Window.orderBy(
    col("salary").desc()
)

In [ ]:
rank_df = employees_df.withColumn(
    "salary_rank",
    rank().over(salary_window)
)

rank_df.show()

In [ ]:
department_window = Window \
    .partitionBy("department") \
    .orderBy(col("salary").desc())

In [ ]:
department_rank_df = employees_df.withColumn(
    "salary_rank",
    rank().over(department_window)
)

department_rank_df.show()

In [ ]:
dense_rank_df = employees_df.withColumn(
    "salary_dense_rank",
    dense_rank().over(department_window)
)

dense_rank_df.show()

In [ ]:
row_window = Window \
    .partitionBy("department") \
    .orderBy(
        col("salary").desc(),
        col("emp_id").asc()
    )

In [ ]:
row_number_df = employees_df.withColumn(
    "row_num",
    row_number().over(row_window)
)

row_number_df.show()

In [ ]:
ranking_window = Window \
    .partitionBy("department") \
    .orderBy(col("salary").desc())

ranking_df = (
    employees_df
    .withColumn(
        "rank",
        rank().over(ranking_window)
    )
    .withColumn(
        "dense_rank",
        dense_rank().over(ranking_window)
    )
    .withColumn(
        "row_number",
        row_number().over(ranking_window)
    )
)

ranking_df.orderBy(
    "department",
    col("salary").desc()
).show()

In [ ]:
top_salary_df = (
    employees_df
    .withColumn(
        "salary_rank",
        dense_rank().over(ranking_window)
    )
    .filter(
        col("salary_rank") <= 2
    )
)

top_salary_df.show()

In [ ]:
top_two_window = Window \
    .partitionBy("department") \
    .orderBy(
        col("salary").desc(),
        col("emp_id").asc()
    )

top_two_df = (
    employees_df
    .withColumn(
        "row_num",
        row_number().over(top_two_window)
    )
    .filter(
        col("row_num") <= 2
    )
)

top_two_df.show()

In [ ]:
sales_data = [
    ("North", "Laptop", "Q1", 50000),
    ("North", "Mobile", "Q1", 30000),
    ("North", "Laptop", "Q2", 60000),
    ("North", "Mobile", "Q2", 35000),

    ("South", "Laptop", "Q1", 45000),
    ("South", "Mobile", "Q1", 25000),
    ("South", "Laptop", "Q2", 55000),
    ("South", "Mobile", "Q2", 40000),

    ("West", "Laptop", "Q1", 70000),
    ("West", "Mobile", "Q1", 45000),
    ("West", "Laptop", "Q2", 75000),
    ("West", "Mobile", "Q2", 50000)
]

sales_columns = [
    "region",
    "product",
    "quarter",
    "sales"
]

sales_df = spark.createDataFrame(
    sales_data,
    sales_columns
)

sales_df.show()

In [ ]:
from pyspark.sql.functions import sum

grouped_df = (
    sales_df
    .groupBy(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
)

grouped_df.show()

In [ ]:
rollup_df = (
    sales_df
    .rollup(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
)

rollup_df.show()

In [ ]:
rollup_df = (
    sales_df
    .rollup(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
    .orderBy(
        "region",
        "product"
    )
)

rollup_df.show()

In [ ]:
three_level_rollup = (
    sales_df
    .rollup(
        "region",
        "product",
        "quarter"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
)

three_level_rollup.show()

In [ ]:
cube_df = (
    sales_df
    .cube(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
)

cube_df.show()

In [ ]:
cube_3d_df = (
    sales_df
    .cube(
        "region",
        "product",
        "quarter"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
)

cube_3d_df.show(100, truncate=False)

In [ ]:
from pyspark.sql.functions import (
    sum,
    avg,
    count,
    max,
    round
)

rollup_analysis_df = (
    sales_df
    .rollup(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales"),
        round(avg("sales"), 2).alias("average_sales"),
        count("*").alias("transactions"),
        max("sales").alias("highest_sale")
    )
)

rollup_analysis_df.show()

In [ ]:
from pyspark.sql.functions import coalesce, lit

readable_rollup_df = (
    sales_df
    .rollup(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales")
    )
    .select(
        coalesce(
            col("region"),
            lit("ALL REGIONS")
        ).alias("region"),

        coalesce(
            col("product"),
            lit("ALL PRODUCTS")
        ).alias("product"),

        "total_sales"
    )
)

readable_rollup_df.show()

In [29]:
from pyspark.sql.functions import (
    sum,
    avg,
    count,
    round,
    coalesce,
    lit,
    col
)

management_report_df = (
    sales_df
    .rollup(
        "region",
        "product"
    )
    .agg(
        sum("sales").alias("total_sales"),
        round(
            avg("sales"), 2
        ).alias("average_sales"),
        count("*").alias("transactions")
    )
    .select(
        coalesce(
            col("region"),
            lit("ALL REGIONS")
        ).alias("region"),

        coalesce(
            col("product"),
            lit("ALL PRODUCTS")
        ).alias("product"),

        "total_sales",
        "average_sales",
        "transactions"
    )
)

management_report_df.show(
    truncate=False
)

+-----------+------------+-----------+-------------+------------+
|region     |product     |total_sales|average_sales|transactions|
+-----------+------------+-----------+-------------+------------+
|North      |Laptop      |110000     |55000.0      |2           |
|ALL REGIONS|ALL PRODUCTS|580000     |48333.33     |12          |
|North      |ALL PRODUCTS|175000     |43750.0      |4           |
|North      |Mobile      |65000      |32500.0      |2           |
|South      |Laptop      |100000     |50000.0      |2           |
|South      |ALL PRODUCTS|165000     |41250.0      |4           |
|South      |Mobile      |65000      |32500.0      |2           |
|West       |Laptop      |145000     |72500.0      |2           |
|West       |ALL PRODUCTS|240000     |60000.0      |4           |
|West       |Mobile      |95000      |47500.0      |2           |
+-----------+------------+-----------+-------------+------------+

